# LANL Cyber 1: data validation and label alignment

This notebook validates the current demonstration feature dataset before any detector is trained. It does not train Isolation Forest, XGBoost, Random Forest, a GNN, or any other model.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_auth_events, load_redteam_events
from src.features import FEATURE_COLUMNS, build_feature_vector

FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'features_sample.parquet'
AUTH_PATH = PROJECT_ROOT / 'data' / 'raw' / 'auth.txt.gz'
REDTEAM_PATH = PROJECT_ROOT / 'data' / 'raw' / 'redteam.txt.gz'

features = pd.read_parquet(FEATURE_PATH)
redteam = load_redteam_events(REDTEAM_PATH)

# A bounded read is enough to inspect status encoding; the full auth archive is not loaded.
auth_status_sample = load_auth_events(AUTH_PATH, max_rows=200_000)
print('Feature rows:', len(features))
print('Red-team rows:', len(redteam))
print('Bounded auth rows inspected:', len(auth_status_sample))

Feature rows: 285
Red-team rows: 749
Bounded auth rows inspected: 200000


## A. Authentication status validation

In [2]:
status_counts = auth_status_sample['status'].value_counts(dropna=False).rename('count').to_frame()
status_counts['proportion'] = status_counts['count'] / len(auth_status_sample)
display(status_counts)

print('Feature status totals:')
display(features[['total_auth_events', 'successful_auth_count', 'failed_auth_count']].sum().to_frame('sum'))
print("Validation result: LANL uses 'Fail', not 'Failure'. features.py currently checks only 'Failure', so failed_auth_count must be fixed before ML.")

,count,proportion
status,,
Success,198593,0.992965
Fail,1407,0.007035


Feature status totals:


,sum
total_auth_events,1353
successful_auth_count,1352
failed_auth_count,0


Validation result: LANL uses 'Fail', not 'Failure'. features.py currently checks only 'Failure', so failed_auth_count must be fixed before ML.


## B-C. Ground truth schema, entity types, and timestamps

LANL's red-team file is the only explicit ground-truth source currently present. Its rows are known compromise events with `timestamp`, `user`, `source_computer`, and `destination_computer`. The feature sample contains `entity_type`, but its `USER_OR_COMPUTER` value shows that some identifiers are ambiguous when inferred from the bounded auth subset.

In [3]:
print('Red-team columns:', redteam.columns.tolist())
print('Red-team schema:')
display(redteam.dtypes.rename('dtype').to_frame())
print('Red-team timestamp range:', int(redteam['timestamp'].min()), 'to', int(redteam['timestamp'].max()))
print('Red-team unique users:', redteam['user'].nunique())
print('Red-team unique source computers:', redteam['source_computer'].nunique())
print('Red-team unique destination computers:', redteam['destination_computer'].nunique())
print('Feature timestamp range:', int(features['timestamp'].min()), 'to', int(features['timestamp'].max()))
print('Feature entity types:')
display(features['entity_type'].value_counts(dropna=False).rename('count').to_frame())

Red-team columns: ['timestamp', 'user', 'source_computer', 'destination_computer']
Red-team schema:


,dtype
timestamp,Int64
user,object
source_computer,object
destination_computer,object


Red-team timestamp range: 150885 to 2557047
Red-team unique users: 104
Red-team unique source computers: 4
Red-team unique destination computers: 301
Feature timestamp range: 150885 to 227052
Feature entity types:


,count
entity_type,
COMPUTER,210
USER,56
USER_OR_COMPUTER,19


## D-F. Align labels without inventing benign negatives

A feature row is directly aligned to known red-team activity when its `(entity, timestamp)` pair matches one of the red-team event's `user`, `source_computer`, or `destination_computer` values at the same timestamp. This supports a positive entity-time indicator. A non-match is left unlabeled: it is not automatically benign, because the red-team file is incomplete ground truth rather than a complete normality annotation.

In [4]:
redteam_keys = {
    (int(event.timestamp), entity)
    for event in redteam.itertuples(index=False)
    for entity in (event.user, event.source_computer, event.destination_computer)
}
features['redteam_entity_match'] = [
    (int(row.timestamp), row.entity) in redteam_keys
    for row in features.itertuples(index=False)
]
# 1 means directly supported by the known red-team source; <NA> means unlabeled, not benign.
features['supervised_label'] = features['redteam_entity_match'].map({True: 1, False: pd.NA}).astype('Int64')

positive_rows = int(features['redteam_entity_match'].sum())
unlabeled_rows = int((~features['redteam_entity_match']).sum())
print('Total feature rows:', len(features))
print('Direct positive rows:', positive_rows)
print('Unlabeled candidate rows:', unlabeled_rows)
print('Unique positive entities:', features.loc[features['redteam_entity_match'], 'entity'].nunique())
print('Unique unlabeled entities:', features.loc[~features['redteam_entity_match'], 'entity'].nunique())
print('Timestamp overlap:', len(set(features['timestamp']) & set(redteam['timestamp'])))
print('Direct-match proportion:', positive_rows / len(features))
display(features[['timestamp', 'entity', 'entity_type', 'redteam_entity_match', 'supervised_label']].head(20))

Total feature rows: 285
Direct positive rows: 45
Unlabeled candidate rows: 240
Unique positive entities: 19
Unique unlabeled entities: 19
Timestamp overlap: 15
Direct-match proportion: 0.15789473684210525


,timestamp,entity,entity_type,redteam_entity_match,supervised_label
0,150885,C1003,COMPUTER,True,1
1,150885,C1173,COMPUTER,False,<NA>
2,150885,C148,COMPUTER,False,<NA>
3,150885,C1493,USER_OR_COMPUTER,False,<NA>
4,150885,C152,USER_OR_COMPUTER,False,<NA>
5,150885,C17693,COMPUTER,True,1
6,150885,C18025,USER_OR_COMPUTER,False,<NA>
7,150885,C2341,USER_OR_COMPUTER,False,<NA>
8,150885,C294,COMPUTER,False,<NA>
9,150885,C305,COMPUTER,False,<NA>


In [5]:
print('Direct positive examples:')
display(features.loc[features['redteam_entity_match'], ['timestamp', 'entity', 'entity_type'] + FEATURE_COLUMNS].head(10))
print('Feature rows around the first known red-team timestamp:')
first_redteam_time = int(redteam['timestamp'].iloc[0])
display(features.loc[features['timestamp'].eq(first_redteam_time), ['timestamp', 'entity', 'redteam_entity_match'] + FEATURE_COLUMNS])

Direct positive examples:


,timestamp,entity,entity_type,total_auth_events,successful_auth_count,failed_auth_count,unique_source_computers,unique_destination_computers,new_destination_count,unique_users,new_edge_count,outgoing_degree,incoming_degree,event_rate
0,150885,C1003,COMPUTER,5,5,0,2,1,1,5,1,1,2,1.0
5,150885,C17693,COMPUTER,3,3,0,1,2,2,1,2,2,0,0.6
16,150885,U620@DOM1,USER,3,3,0,1,2,2,1,2,2,1,0.6
24,151036,C17693,COMPUTER,4,4,0,1,3,3,2,3,3,0,0.8
28,151036,C305,COMPUTER,1,1,0,1,1,0,1,0,0,1,0.2
37,151036,U748@DOM1,USER,8,8,0,4,4,4,1,4,4,1,1.6
43,151648,C17693,COMPUTER,1,1,0,1,1,1,1,1,1,0,0.2
52,151648,C728,COMPUTER,24,24,0,3,6,5,2,5,6,3,4.8
56,151648,U748@DOM1,USER,12,12,0,7,6,4,1,4,6,1,2.4
58,151993,C1173,COMPUTER,2,2,0,2,1,0,2,0,1,2,0.4


Feature rows around the first known red-team timestamp:


,timestamp,entity,redteam_entity_match,total_auth_events,successful_auth_count,failed_auth_count,unique_source_computers,unique_destination_computers,new_destination_count,unique_users,new_edge_count,outgoing_degree,incoming_degree,event_rate
0,150885,C1003,True,5,5,0,2,1,1,5,1,1,2,1.0
1,150885,C1173,False,3,3,0,2,1,1,2,1,1,2,0.6
2,150885,C148,False,1,1,0,1,1,1,1,1,1,0,0.2
3,150885,C1493,False,0,0,0,0,0,0,0,0,0,0,0.0
4,150885,C152,False,0,0,0,0,0,0,0,0,0,0,0.0
5,150885,C17693,True,3,3,0,1,2,2,1,2,2,0,0.6
6,150885,C18025,False,0,0,0,0,0,0,0,0,0,0,0.0
7,150885,C2341,False,0,0,0,0,0,0,0,0,0,0,0.0
8,150885,C294,False,5,5,0,1,2,2,2,2,2,0,1.0
9,150885,C305,False,1,1,0,1,1,1,1,1,1,0,0.2


## G-I. Duplicate and temporal-leakage checks

In [6]:
print('Exact duplicate feature rows:', int(features.duplicated().sum()))
print('Duplicate entity/timestamp keys:', int(features.duplicated(['entity', 'timestamp']).sum()))
print('Window end equals prediction timestamp:', bool(features['window_end'].eq(features['timestamp']).all()))
print('Window starts before or at prediction timestamp:', bool((features['window_start'] <= features['timestamp']).all()))
print('Any feature window extends after its prediction timestamp:', bool((features['window_end'] > features['timestamp']).any()))

# The bounded status sample may not cover the feature timestamps. Only run an
# independent recomputation when its range actually covers the selected row.
check_row = features.iloc[0]
status_sample_covers_row = (
    auth_status_sample['timestamp'].min() <= check_row['window_start']
    and auth_status_sample['timestamp'].max() >= check_row['window_end']
)
print('Bounded auth sample covers selected feature row:', status_sample_covers_row)
if status_sample_covers_row:
    auth_until_prediction = auth_status_sample[
        auth_status_sample['timestamp'] <= check_row['timestamp']
    ]
    recomputed = build_feature_vector(
        auth_until_prediction,
        check_row['entity'],
        int(check_row['window_start']),
        int(check_row['window_end']),
    )
    leakage_columns = FEATURE_COLUMNS
    leakage_check = {
        column: recomputed[column] == check_row[column]
        for column in leakage_columns
    }
    print('Independent future-data leakage check:', all(leakage_check.values()))
else:
    print('Independent recomputation skipped: bounded status sample does not cover the feature timestamp.')
print('Feature code rule: prior history uses timestamp < window_start; active events use window_start <= timestamp <= window_end.')

Exact duplicate feature rows: 0
Duplicate entity/timestamp keys: 0
Window end equals prediction timestamp: True
Window starts before or at prediction timestamp: True
Any feature window extends after its prediction timestamp: False
Bounded auth sample covers selected feature row: False
Independent recomputation skipped: bounded status sample does not cover the feature timestamp.
Feature code rule: prior history uses timestamp < window_start; active events use window_start <= timestamp <= window_end.


## Temporal split feasibility and recommendation

The current sample has only 15 distinct timestamps, all selected from known red-team activity, so it is too small and too targeted for a meaningful train/validation/test experiment. A future split can be made chronologically only after generating a broader feature dataset over normal authentication activity. Random row splitting is inappropriate because overlapping windows from nearby timestamps and repeated entities would leak temporal context.

In [7]:
print('Distinct feature timestamps:', features['timestamp'].nunique())
print('Feature timestamp values:', sorted(features['timestamp'].unique().tolist()))
print('')
print('Recommendation:')
print('1. Keep features_sample.parquet as a smoke-test and alignment-validation artifact, not the first ML dataset.')
print('2. Build a broader time-ordered feature dataset from auth events, including ordinary activity and explicit empty/low-activity windows.')
print('3. Treat redteam_entity_match=1 as a positive entity-time indicator; treat non-matches as unlabeled unless a defensible benign sampling rule is documented.')
print('4. Start with unsupervised anomaly detection or a hybrid workflow. Supervised learning requires reliable negatives and broader temporal coverage.')
print('5. Notebook 05 should first generate the chronological dataset and split by time, then train/evaluate a detector only after the label and status issues are fixed.')
print("6. Fix failed_auth_count to recognize LANL's 'Fail' status before ML.")

Distinct feature timestamps: 15
Feature timestamp values: [150885, 151036, 151648, 151993, 153792, 155219, 155399, 155460, 155591, 156658, 210086, 210294, 210312, 218418, 227052]

Recommendation:
1. Keep features_sample.parquet as a smoke-test and alignment-validation artifact, not the first ML dataset.
2. Build a broader time-ordered feature dataset from auth events, including ordinary activity and explicit empty/low-activity windows.
3. Treat redteam_entity_match=1 as a positive entity-time indicator; treat non-matches as unlabeled unless a defensible benign sampling rule is documented.
4. Start with unsupervised anomaly detection or a hybrid workflow. Supervised learning requires reliable negatives and broader temporal coverage.
5. Notebook 05 should first generate the chronological dataset and split by time, then train/evaluate a detector only after the label and status issues are fixed.
6. Fix failed_auth_count to recognize LANL's 'Fail' status before ML.
